# Capstone Report — 5-min demo outline + 2 shareable write-ups

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ku-ro-wa/flyrank-ml-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

If I had 5 minutes to demo the project, this would be a viable outline.


> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 0. 5-minute demo outline

*The brief this answers: **growth / recovery / momentum prediction** — from a page's own warehouse daily facts, which pages are about to move, in time to reprioritise the week? One minute per beat.*

## 1. Question

Per content page, forecast next-30-day GSC clicks and GA4 sessions from time windows I build myself over the daily fact table, with a strict leakage audit.

The decision it feeds: a content team's weekly ordering of which *moving* pages to look at first. Success is defined as beating the "assume the recent rate continues" naive baseline on the same split — not as hitting an absolute accuracy number.

## 2. Method

*78.8M daily rows → 5.68M weekly-anchored `(page, anchor_date)` windows: 90-day rolling features, 30-day forward target.

Temporal split (train early anchors, test later) with a 30-day gap so no training row's target window straddles the cutoff.

Leakage audit: target encoder fit on train only, feature windows end strictly before target windows, plus a client-holdout split to confirm the model isn't memorising client identity.

Four model families kept side by side against naive: Tweedie regressor, hurdle model (classifier `P(active)` × conditional regressor), a validation-routed blend, and a LambdaRank ranker.

## 3. One chart

`figures/per_bucket_mae.png` — MAE by prior-90-day activity bucket, shipped model vs. naive, both targets. Talk through it live: the near-zero bucket is roughly half of the GSC test set and *no method has ranking skill there*; the model's advantage is confined to mid- and high-activity content where there is movement to read.

## 4. One honest result

The headline is *not* "we beat the baseline." For **GA4 sessions** the blended hurdle model wins every pooled metric (MAE 4.12 vs 4.50, RMSE 114.1 vs 121.4, Spearman 0.698 vs 0.697) and all three on active content.

For **GSC clicks** no trained model clears naive on the full pooled set — naive keeps RMSE 13.91 and Spearman 0.712. That gap is a limitation, not a bug: for the low-activity long tail that dominates GSC, the ceiling is per-entity signal density, not data volume or window length (97.6% of content that is near-zero over 90 days is also near-zero over its own trailing 30 days).

## 5. One recommendation

Ship the split: **blend for GA4, naive alone for GSC.** Deliver it as a weekly-refreshed, tiered, human-gated action queue — one row per page, a confidence tier (`model_driven` / `naive_fallback` / `monitor_only`), a coarse momentum flag, a plain-language reason, and a `needs_human_review` gate.

Never present the GSC order as model skill to a client. Adopt the 60-day intake window for coverage of pages too young for a 90-day lookback, flagged lower-confidence.

## 6. If asked "what would you do next"

Rolling-origin cross-validation for the router (a single validation slice picked the wrong activity bucket in both directions).

A breakout classifier for the dead tail instead of a count regression.

And rolling query-level snapshots — `content_visible_query_count` correlates 0.57 with GSC naive's absolute error but can't be joined leakage-free under the current single-export-window data.

## 7. Two shareable cuts

### Short social post (methodology)
> Spent my capstone forecasting 30-day content performance, and the most useful result was a negative one.
>
> Setup: 78.8M daily rows from a warehouse fact table → 5.68M weekly-anchored windows I built myself (90-day rolling features, 30-day forward target). Temporal split with a 30-day gap so no training row's target window straddles the cutoff. Target encoder fit on train only. A client-holdout split to check the model wasn't just memorising client identity.
>
> The discipline I kept: every model — four families, from a Tweedie regressor to a LambdaRank ranker — is scored on the *same* split against a naive "assume the recent rate continues" baseline, on three metrics at once (MAE, RMSE, rank correlation), pooled *and* broken out by activity level, so a win on the near-zero mass can't hide a loss on the content that actually matters.
>
> Result: the model genuinely beats naive for one target (GA4 sessions). For the other (GSC clicks) it doesn't — and profiling showed why: ~half the pages are near-zero at every window length, so there's no signal to find; it's not a data-volume problem. The shipping recommendation is therefore split — model for one target, naive for the other — delivered as a human-gated weekly queue.
>
> Reporting the baseline honestly was worth more than another point of accuracy.


### Employer-facing summary (What I built, on what data, what it showed)
> I built a future-window forecasting pipeline that predicts a content page's next-30-day Google Search Console clicks and GA4 sessions, turning 78.8M daily warehouse rows into 5.68M leakage-audited training windows and training four model families (Tweedie, hurdle, a validation-routed blend, and a LambdaRank ranker) against a naive baseline on a strict temporal split. It showed that the blended hurdle model is a real, repeatable win for GA4 sessions on every metric, while for GSC clicks no model beats the naive baseline — because roughly half the content is near-zero at every window length, a per-entity signal-density floor rather than a data-volume problem. The deliverable is a shipping recommendation split by target plus a weekly, tiered, human-gated action queue, with the negative GSC result reported as prominently as the positive GA4 one.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.